In [ ]:
# Cell-DINO SSL training from a 4-channel NumPy array (inline Python, no subprocess)
from pathlib import Path
import os
import sys
import numpy as np
import torch

In [ ]:
# Paths and run settings
REPO_DIR = Path("/Users/svo/source/dinov2")
NPY_PATH = Path("/scratch/svo/morpho_analysis/toy_dataset/MIPFTMA14/aveolar/processed_img_with_mask.npy")
CONFIG_FILE = REPO_DIR / "dinov2/configs/train/cell_dino/vitl16_npy128.yaml"
OUTPUT_DIR = REPO_DIR / "outputs/cell_dino_npy128_run1"

# Optional overrides for quick sanity checks
EPOCHS = 5
NUM_WORKERS = 4
BATCH_SIZE_PER_GPU = 8

assert REPO_DIR.exists(), f"Missing repo dir: {REPO_DIR}"
assert NPY_PATH.exists(), f"Missing dataset: {NPY_PATH}"
assert CONFIG_FILE.exists(), f"Missing config: {CONFIG_FILE}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

arr = np.load(NPY_PATH, mmap_mode="r")
print("Dataset shape:", arr.shape, "dtype:", arr.dtype, "min:", arr.min(), "max:", arr.max())
if arr.ndim != 4 or arr.shape[1] != 4:
    raise ValueError(f"Expected [N,4,H,W], got {arr.shape}")

In [ ]:
# Build train opts (same format as train.py CLI opts)
dataset_opt = f"train.dataset_path=NPYCells:split=ALL:root={NPY_PATH}"
opts = [
    dataset_opt,
    f"optim.epochs={EPOCHS}",
    f"train.num_workers={NUM_WORKERS}",
    f"train.batch_size_per_gpu={BATCH_SIZE_PER_GPU}",
]
for item in opts:
    print(item)

In [ ]:
# Run training inline using code path from dinov2/train/train.py
from types import SimpleNamespace

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from dinov2.utils.config import setup
from dinov2.train.ssl_meta_arch import SSLMetaArch
from dinov2.train.train import do_train, do_test
from dinov2.fsdp import FSDPCheckpointer
import dinov2.distributed as distributed

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this training script.")
if distributed.is_enabled():
    raise RuntimeError(
        "Distributed process group is already initialized in this kernel. "
        "Restart kernel before running this cell again."
    )

args = SimpleNamespace(
    config_file=str(CONFIG_FILE),
    no_resume=False,
    eval_only=False,
    eval="",
    opts=list(opts),
    output_dir=str(OUTPUT_DIR),
    seed=0,
)

cfg = setup(args)
model = SSLMetaArch(cfg).to(torch.device("cuda"))
model.prepare_for_distributed_training()

if args.eval_only:
    iteration = (
        FSDPCheckpointer(model, save_dir=cfg.train.output_dir)
        .resume_or_load(cfg.MODEL.WEIGHTS, resume=not args.no_resume)
        .get("iteration", -1) + 1
    )
    do_test(cfg, model, f"manual_{iteration}")
else:
    do_train(cfg, model, resume=not args.no_resume)

print(f"Training completed. Outputs saved to: {OUTPUT_DIR}")